# 🧠 Brain Tumor MRI Image Classification

## Clinical AI Project Overview

This project builds an end-to-end deep learning pipeline for classifying brain MRI scans into four classes: **glioma**, **meningioma**, **notumor**, and **pituitary**. The workflow covers dataset understanding, preprocessing, augmentation, custom CNN training, transfer learning with EfficientNetB0 and MobileNetV2, model evaluation, Grad-CAM explainability, model comparison, and deployment preparation.

## Clinical Motivation

Brain tumor diagnosis from MRI is time-sensitive and clinically significant. A robust image classifier can support radiologists by providing fast second-opinion screening, especially in settings with limited specialist availability. In this project, special attention is given to recall for malignant tumor classes such as glioma, because missing a malignant case can be more dangerous than raising a false alarm.

## Notebook Goals

- Train on **Google Colab GPU**
- Use a **Google Drive-mounted dataset**
- Build:
  - Custom CNN from scratch
  - EfficientNetB0 transfer learning model
  - MobileNetV2 transfer learning model
  - InceptionV3 transfer learning model
- Evaluate with:
  - Accuracy/loss curves
  - Classification report
  - Confusion matrix
  - ROC-AUC curves
  - Grad-CAM
- Save all deployable artifacts for Streamlit


In [1]:
import os 
print(os.listdir("/kaggle/input"))

['datasets']


In [2]:
# =========================
# PHASE 0A: KAGGLE SETUP
# =========================

!pip install -q tf-keras-vis

import os
import json
import time
import random
import shutil
import zipfile
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2, InceptionV3
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.utils import plot_model

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score, precision_score, recall_score

from tf_keras_vis.gradcam import Gradcam
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("GPU Devices:", tf.config.list_physical_devices('GPU'))

# =========================
# KAGGLE DATASET PATH SETUP
# =========================
KAGGLE_INPUT_ROOT = "/kaggle/input"

print("\nAvailable datasets inside /kaggle/input:")
for item in sorted(os.listdir(KAGGLE_INPUT_ROOT)):
    print("-", item)

# CHANGE THIS to your actual dataset folder name seen above
DATASET_ROOT = "/kaggle/input/datasets"

TRAIN_DIR = os.path.join(DATASET_ROOT, "Training")
TEST_DIR = os.path.join(DATASET_ROOT, "Testing")

if not os.path.exists(TRAIN_DIR):
    raise FileNotFoundError(f"{TRAIN_DIR} not found. Check your Kaggle dataset folder name and structure.")
if not os.path.exists(TEST_DIR):
    raise FileNotFoundError(f"{TEST_DIR} not found. Check your Kaggle dataset folder name and structure.")

print("\nResolved DATASET_ROOT:", DATASET_ROOT)
print("Resolved TRAIN_DIR:", TRAIN_DIR)
print("Resolved TEST_DIR:", TEST_DIR)
print("Train classes:", sorted(os.listdir(TRAIN_DIR)))
print("Test classes:", sorted(os.listdir(TEST_DIR)))

# Save all generated outputs in Kaggle working directory
BASE_WORK_DIR = "/kaggle/working"
MODELS_DIR = os.path.join(BASE_WORK_DIR, "models")
PLOTS_DIR = os.path.join(BASE_WORK_DIR, "plots")
SCREENSHOTS_DIR = os.path.join(BASE_WORK_DIR, "screenshots")
DATA_DIR = os.path.join(BASE_WORK_DIR, "data")
EXPORTS_DIR = os.path.join(BASE_WORK_DIR, "exports")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(SCREENSHOTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(EXPORTS_DIR, exist_ok=True)

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

CUSTOM_CNN_EPOCHS = 50
TRANSFER_PHASE1_EPOCHS = 20
TRANSFER_PHASE2_EPOCHS = 30

print("\nKaggle environment ready.")
print("All saved files should go under /kaggle/working so they appear in Output.")
print("MODELS_DIR:", MODELS_DIR)
print("PLOTS_DIR:", PLOTS_DIR)
print("EXPORTS_DIR:", EXPORTS_DIR)

def export_all_artifacts(zip_name='brain_tumor_mri_artifacts.zip'):
    zip_path = os.path.join(EXPORTS_DIR, zip_name)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for folder in [MODELS_DIR, PLOTS_DIR, SCREENSHOTS_DIR, DATA_DIR]:
            if os.path.exists(folder):
                for root, _, filenames in os.walk(folder):
                    for filename in filenames:
                        full_path = os.path.join(root, filename)
                        arcname = os.path.relpath(full_path, BASE_WORK_DIR)
                        zf.write(full_path, arcname=arcname)
    print(f"Created zip: {zip_path}")
    return zip_path

def save_training_artifacts_summary():
    summary = {
        'models_present': sorted(os.listdir(MODELS_DIR)) if os.path.exists(MODELS_DIR) else [],
        'plots_present': sorted(os.listdir(PLOTS_DIR)) if os.path.exists(PLOTS_DIR) else [],
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    summary_path = os.path.join(EXPORTS_DIR, 'artifact_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"Saved {summary_path}")
    return summary

^C


2026-06-20 09:57:10.096912: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781949430.318557      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781949430.387635      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781949430.957205      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781949430.957245      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781949430.957247      58 computation_placer.cc:177] computation placer alr

ModuleNotFoundError: No module named 'tf_keras_vis'

In [ ]:
# Quick dataset structure check
print("TRAIN_DIR exists:", os.path.exists(TRAIN_DIR))
print("TEST_DIR exists:", os.path.exists(TEST_DIR))

print("\nTrain subfolders:", sorted(os.listdir(TRAIN_DIR)))
print("Test subfolders:", sorted(os.listdir(TEST_DIR)))


## Environment Notes

This notebook is designed to run in **Kaggle Notebooks**. The dataset is expected to be attached through the Kaggle **Add Input** panel and will appear under `/kaggle/input/...`, so no manual upload or unzip step is needed.

We also fix seeds across Python, NumPy, and TensorFlow. This does not guarantee perfect determinism in all operations, but it improves reproducibility and makes model comparison more reliable.

All generated artifacts should be saved inside `/kaggle/working/models`, `/kaggle/working/plots`, and `/kaggle/working/exports`, because files written under `/kaggle/working` appear in the Kaggle Output panel for download.

## PHASE 0B — Local Virtual Environment for Streamlit

Use the following commands locally after training is finished and all artifacts are saved.


In [ ]:
venv_guide = r"""
# Windows
python -m venv brain_tumor_env
brain_tumor_env\Scripts\activate

# macOS / Linux
python3 -m venv brain_tumor_env
source brain_tumor_env/bin/activate

pip install -r requirements.txt
streamlit run app.py
"""
print(venv_guide)


## PHASE 1 — Dataset Understanding

The dataset contains four classes commonly used in brain MRI classification benchmarks:
- **glioma** — often malignant and clinically high-risk
- **meningioma** — often benign but still clinically important
- **notumor** — healthy brain MRI
- **pituitary** — tumor around the pituitary gland at the brain base

A clear dataset audit is important before training. In medical AI, weak dataset understanding often causes misleading performance claims, label leakage, or hidden imbalance that later hurts clinical trust.


In [ ]:
# Step 1 — Dataset loading and class counts
def count_images_in_directory(base_dir):
    records = []
    total = 0
    for class_name in sorted(os.listdir(base_dir)):
        class_path = os.path.join(base_dir, class_name)
        if os.path.isdir(class_path):
            image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            count = len(image_files)
            total += count
            records.append({"class_name": class_name, "count": count})
    df = pd.DataFrame(records)
    df["percentage_share"] = (df["count"] / df["count"].sum() * 100).round(2)
    return df, total

train_counts_df, train_total = count_images_in_directory(TRAIN_DIR)
test_counts_df, test_total = count_images_in_directory(TEST_DIR)

summary_df = train_counts_df.merge(
    test_counts_df.rename(columns={"count": "test_count", "percentage_share": "test_percentage"}),
    on="class_name",
    how="outer"
).fillna(0)

summary_df["total_count"] = summary_df["count"] + summary_df["test_count"]
summary_df["overall_percentage"] = (summary_df["total_count"] / summary_df["total_count"].sum() * 100).round(2)
summary_df["suggested_train_pct"] = 80
summary_df["suggested_val_pct"] = 10
summary_df["suggested_test_pct"] = 10

summary_df = summary_df.rename(columns={"count": "train_count"})
summary_df


This table gives the first sanity check of class coverage across the project. If one class has much fewer examples than the others, the model may learn a biased decision boundary and underperform on clinically important tumor types.

From a clinical perspective, imbalance matters because an underrepresented malignant class may be predicted less often, creating a dangerous pattern of false reassurance.


In [ ]:
# Visual sample grid
plt.style.use("dark_background")
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(16, 14), facecolor="#1A1A1A")

sample_paths = {}
for idx, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(TRAIN_DIR, class_name)
    files = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    chosen = random.sample(files, min(4, len(files)))
    sample_paths[class_name] = chosen

    for j in range(4):
        ax = axes[idx, j]
        ax.set_facecolor("#1A1A1A")
        if j < len(chosen):
            img = cv2.cvtColor(cv2.imread(chosen[j]), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        ax.axis("off")
        if j == 0:
            ax.set_title(class_name.upper(), color="white", fontsize=13, loc="left")

plt.suptitle("Sample MRI Images Per Class", fontsize=18, color="white", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "02_sample_images_grid.png"), dpi=300, bbox_inches="tight")
plt.show()


The sample grid helps verify that the classes are visually distinct and that labels appear plausible. In medical imaging work, this step also helps catch common data issues such as mislabeled folders, duplicate images, excessive artifacts, or non-MRI content mixed into the dataset.

Visual auditing is especially useful before transfer learning, because pretrained backbones can amplify dataset shortcuts if the dataset is noisy.


In [ ]:
# Resolution consistency check
all_train_images = []
for class_name in CLASS_NAMES:
    class_dir = os.path.join(TRAIN_DIR, class_name)
    all_train_images.extend([os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

sampled_images = random.sample(all_train_images, min(20, len(all_train_images)))
shape_records = []

for path in sampled_images:
    img = cv2.imread(path)
    shape_records.append({
        "file_name": os.path.basename(path),
        "class_name": Path(path).parent.name,
        "shape": img.shape if img is not None else None
    })

shape_df = pd.DataFrame(shape_records)
shape_df


Resolution consistency affects preprocessing quality and training stability. If the source images vary widely in size, the model may see inconsistent anatomical detail before resizing, which can alter how tumor boundaries or tissue patterns appear.

This is one reason standardized resizing is essential later in the pipeline.


In [ ]:
# Class distribution bar chart
plot_df = summary_df[["class_name", "total_count"]].sort_values("total_count", ascending=False)

plt.figure(figsize=(10, 6), facecolor="#121212")
ax = sns.barplot(data=plot_df, x="class_name", y="total_count", palette="deep")
plt.title("Overall Class Distribution", fontsize=16, color="white")
plt.xlabel("Class", color="white")
plt.ylabel("Image Count", color="white")
plt.xticks(color="white")
plt.yticks(color="white")
ax.set_facecolor("#121212")

for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', color='white', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "01_class_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()


This class distribution plot makes imbalance easy to inspect visually. In a recruiter or evaluator setting, this chart shows that the project does not treat model training as a black box; instead, it explicitly checks whether data composition could distort performance.

For medical classification, this matters because high overall accuracy can still hide poor sensitivity on a smaller but clinically critical class.


In [ ]:
# Step 2 — Class imbalance check
max_count = summary_df["total_count"].max()
min_count = summary_df["total_count"].min()
imbalance_ratio = round(max_count / min_count, 3)

print("Class imbalance ratio (max/min):", imbalance_ratio)

train_class_counts = train_counts_df.set_index("class_name")["train_count"] if "train_count" in train_counts_df.columns else train_counts_df.set_index("class_name")["count"]
train_labels_for_weights = []

for class_idx, class_name in enumerate(CLASS_NAMES):
    count = int(train_counts_df.loc[train_counts_df["class_name"] == class_name, "count"].values[0])
    train_labels_for_weights.extend([class_idx] * count)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels_for_weights),
    y=train_labels_for_weights
)

class_weights = {i: float(w) for i, w in enumerate(class_weights_values)}
class_weights


The imbalance ratio quantifies how uneven the class distribution is. Even moderate imbalance can matter in healthcare AI because the model may learn to favor the most common class, which increases the risk of missed pathology in underrepresented groups.

To reduce that risk, we compute **class weights** and use them during training. That gives larger loss penalties to classes with fewer samples and makes the optimization process more clinically responsible.


## PHASE 2 — Data Preprocessing

Preprocessing standardizes the MRI inputs before they reach the neural network. This is essential because CNNs expect fixed-size tensors, while real datasets often contain varying image dimensions, intensity ranges, and formatting differences.


In [ ]:
# Step 3 — Reusable preprocessing function
def preprocess_image(path, target_size=(224, 224)):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size)
    img = img.astype(np.float32) / 255.0
    return img

# Quick test
sample_img = preprocess_image(sampled_images[0])
print("Processed image shape:", sample_img.shape)
print("Pixel range:", sample_img.min(), sample_img.max())


We resize images to **224×224** because that is the standard input size for several ImageNet-pretrained backbones, including EfficientNetB0 and MobileNetV2. Using this size keeps the pipeline compatible with transfer learning while preserving enough spatial detail for tumor pattern recognition.

We also normalize pixel intensities to the range **[0, 1]**. This improves gradient stability, helps optimization converge faster, and reduces the chance that large raw pixel values dominate training updates.


In [ ]:
# Step 4 — Train/Validation/Test split using image_dataset_from_directory
full_train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    validation_split=0.1111,
    subset='training',
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    validation_split=0.1111,
    subset='validation',
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

class_names_from_ds = full_train_ds.class_names
print("Class names from dataset:", class_names_from_ds)

train_ds = full_train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


Here, the training folder is internally split into training and validation subsets, while the external testing folder is kept fully separate for final evaluation. This is a strong practice because the test set remains untouched during model development.

The use of `.cache()`, `.shuffle()`, and `.prefetch()` improves throughput. In particular, **prefetching** overlaps data loading with GPU computation, which reduces idle GPU time and speeds up training in Colab.


## PHASE 3 — Data Augmentation

Medical image datasets are often limited compared with natural image datasets. Augmentation helps the model generalize by exposing it to realistic variations such as orientation changes, contrast shifts, and small spatial translations that can arise from scanner position, acquisition protocol, or patient movement.


In [ ]:
# Step 5 — Augmentation strategy
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentation_pipeline")

for images, labels in train_ds.take(1):
    sample_original = images[0]
    break

fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor="#111111")
axes = axes.flatten()

axes[0].imshow(sample_original.numpy().astype("uint8"))
axes[0].set_title("Original", color="white")
axes[0].axis("off")

for i in range(1, 7):
    aug_img = augmentation(tf.expand_dims(sample_original, axis=0), training=True)[0].numpy().astype("uint8")
    axes[i].imshow(aug_img)
    axes[i].set_title(f"Augmented {i}", color="white")
    axes[i].axis("off")

axes[7].axis("off")
plt.suptitle("MRI Augmentation Preview", color="white", fontsize=18)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "03_augmented_samples.png"), dpi=300, bbox_inches="tight")
plt.show()


This visualization is important because evaluators can directly verify that augmentation is functioning correctly. In medical imaging, augmentation must be meaningful rather than aggressive; extreme distortion can damage anatomy and create unrealistic training samples.

Used properly, augmentation improves robustness to scanner angle, contrast variation, and small positional shifts while preserving medically relevant structures.


## PHASE 4 — Custom CNN From Scratch

A custom CNN serves as the project baseline. It shows that we can design a task-specific architecture from first principles rather than relying only on pretrained models.


In [ ]:
# Normalization mapping for scratch model
def normalize_ds(ds):
    return ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTOTUNE)

train_ds_norm = normalize_ds(train_ds).prefetch(AUTOTUNE)
val_ds_norm = normalize_ds(val_ds).prefetch(AUTOTUNE)
test_ds_norm = normalize_ds(test_ds).prefetch(AUTOTUNE)

# Step 6 — Custom CNN Architecture
def build_brain_tumor_cnn(input_shape=(224, 224, 3), num_classes=4):
    inputs = keras.Input(shape=input_shape, name="input_layer")
    x = augmentation(inputs)

    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name="BrainTumorCNN")
    return model

custom_cnn = build_brain_tumor_cnn()
custom_cnn.summary()

plot_model(custom_cnn, to_file=os.path.join(PLOTS_DIR,"custom_cnn_architecture.png"), show_shapes=True, show_dtype=True)


This architecture uses progressively deeper convolutional blocks to learn increasingly abstract MRI features. Early layers capture edges and textures, while deeper layers learn more tumor-specific structural patterns.

**BatchNormalization** stabilizes training by reducing internal covariate shift. **Dropout** reduces overfitting by preventing the dense layers from depending too heavily on any one feature pathway. **GlobalAveragePooling2D** is used instead of Flatten because it reduces parameter count and often generalizes better, especially on moderate-sized medical datasets. The final **Softmax** layer is appropriate because the problem has four mutually exclusive classes.


In [ ]:
# Step 7 — Compile and train custom CNN
custom_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

custom_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
    os.path.join(MODELS_DIR, 'custom_cnn_best.h5'),
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
]

start_time = time.time()
history_cnn = custom_cnn.fit(
    train_ds_norm,
    validation_data=val_ds_norm,
    epochs=CUSTOM_CNN_EPOCHS,
    callbacks=custom_callbacks,
    class_weight=class_weights,
    verbose=1
)
cnn_train_time = time.time() - start_time
print("Custom CNN training time (sec):", round(cnn_train_time, 2))


Each callback serves a practical purpose. **EarlyStopping** prevents unnecessary overtraining once validation performance stops improving, similar to stopping a treatment trial when no additional clinical benefit is appearing. **ModelCheckpoint** ensures the best-performing version is saved, not just the last one. **ReduceLROnPlateau** lowers the learning rate when progress slows, allowing finer optimization near a minimum.

These callbacks improve both efficiency and reliability, especially in Colab where training resources are limited.


## PHASE 5 — Transfer Learning

Transfer learning is especially powerful for medical imaging when the dataset is moderate in size. Although MRI images differ from natural photographs, pretrained CNN backbones still provide transferable low-level features such as edges, contours, textures, and spatial primitives.


In [ ]:
# Utility dataset preprocessors for pretrained backbones
def apply_preprocess(ds, preprocess_fn):
    return ds.map(lambda x, y: (preprocess_fn(tf.cast(x, tf.float32)), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

train_ds_eff = apply_preprocess(train_ds, efficientnet_preprocess)
val_ds_eff = apply_preprocess(val_ds, efficientnet_preprocess)
test_ds_eff = apply_preprocess(test_ds, efficientnet_preprocess)

train_ds_mob = apply_preprocess(train_ds, mobilenet_preprocess)
val_ds_mob = apply_preprocess(val_ds, mobilenet_preprocess)
test_ds_mob = apply_preprocess(test_ds, mobilenet_preprocess)

train_ds_inc = apply_preprocess(train_ds, inception_preprocess)
val_ds_inc = apply_preprocess(val_ds, inception_preprocess)
test_ds_inc = apply_preprocess(test_ds, inception_preprocess)


In [ ]:
# Step 8 — EfficientNetB0
def build_efficientnet_model(input_shape=(224, 224, 3), num_classes=4):
    base_model = EfficientNetB0(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = augmentation(inputs)
    x = efficientnet_preprocess(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name="EfficientNetB0_BrainTumor")
    return model, base_model

efficientnet_model, efficientnet_base = build_efficientnet_model()

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

eff_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
    os.path.join(MODELS_DIR, 'efficientnetb0_best.h5'),
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7)
]

start_time = time.time()
history_eff_phase1 = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TRANSFER_PHASE1_EPOCHS,
    callbacks=eff_callbacks,
    class_weight=class_weights,
    verbose=1
)

efficientnet_base.trainable = True
for layer in efficientnet_base.layers[:-20]:
    layer.trainable = False

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_eff_phase2 = efficientnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TRANSFER_PHASE2_EPOCHS,
    callbacks=eff_callbacks,
    class_weight=class_weights,
    verbose=1
)

eff_train_time = time.time() - start_time
print("EfficientNetB0 total training time (sec):", round(eff_train_time, 2))


ImageNet initialization helps because early CNN layers learn general visual primitives that remain useful even outside natural photography. In MRI, edges, textures, shape transitions, and local patterns still matter, so these low-level filters transfer surprisingly well.

Fine-tuning only the top portion of the network is a safer strategy than unfreezing everything immediately. It preserves broadly useful feature extractors while allowing the more task-specific upper layers to adapt to the medical domain.


In [ ]:
# Step 9 — MobileNetV2
def build_mobilenet_model(input_shape=(224, 224, 3), num_classes=4):
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = augmentation(inputs)
    x = mobilenet_preprocess(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name="MobileNetV2_BrainTumor")
    return model, base_model

mobilenet_model, mobilenet_base = build_mobilenet_model()

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mob_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
    os.path.join(MODELS_DIR, 'mobilenetv2_best.h5'),
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7)
]

start_time = time.time()
history_mob_phase1 = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TRANSFER_PHASE1_EPOCHS,
    callbacks=mob_callbacks,
    class_weight=class_weights,
    verbose=1
)

mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-20]:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_mob_phase2 = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TRANSFER_PHASE2_EPOCHS,
    callbacks=mob_callbacks,
    class_weight=class_weights,
    verbose=1
)

mob_train_time = time.time() - start_time
print("MobileNetV2 total training time (sec):", round(mob_train_time, 2))


MobileNetV2 is attractive for real-world deployment because it is lightweight and efficient. Its depthwise separable convolutions reduce parameter count and computation cost, which is valuable for low-resource hospital systems, edge deployment, or rapid inference in screening settings.


In [ ]:
# Step 10 — InceptionV3 (optional additional architecture)
def build_inception_model(input_shape=(224, 224, 3), num_classes=4):
    base_model = InceptionV3(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = augmentation(inputs)
    x = inception_preprocess(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name="InceptionV3_BrainTumor")
    return model, base_model

inception_model, inception_base = build_inception_model()

inception_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

inc_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
    os.path.join(MODELS_DIR, 'inceptionv3_best.h5'),
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7)
]

start_time = time.time()
history_inc_phase1 = inception_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TRANSFER_PHASE1_EPOCHS,
    callbacks=inc_callbacks,
    class_weight=class_weights,
    verbose=1
)

inception_base.trainable = True
for layer in inception_base.layers[:-20]:
    layer.trainable = False

inception_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_inc_phase2 = inception_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TRANSFER_PHASE2_EPOCHS,
    callbacks=inc_callbacks,
    class_weight=class_weights,
    verbose=1
)

inc_train_time = time.time() - start_time
print("InceptionV3 total training time (sec):", round(inc_train_time, 2))


InceptionV3 provides another useful comparison point because its multi-scale convolution design can capture patterns at different receptive fields. In medical imaging, that may help when tumor appearance varies in size, shape, and surrounding context.


## PHASE 6 — Model Evaluation

Model evaluation in healthcare must go beyond accuracy. We need class-wise precision, recall, F1-score, confusion matrices, and explainability tools to judge whether the system is clinically trustworthy.


In [ ]:
# Utility: combine phase histories
def merge_histories(*histories):
    merged = defaultdict(list)
    for hist in histories:
        for k, v in hist.history.items():
            merged[k].extend(v)
    return dict(merged)

cnn_history = history_cnn.history
eff_history = merge_histories(history_eff_phase1, history_eff_phase2)
mob_history = merge_histories(history_mob_phase1, history_mob_phase2)
inc_history = merge_histories(history_inc_phase1, history_inc_phase2)


In [ ]:
# Step 11 — Training history plots
plt.style.use('dark_background')

def plot_training_history(history_dict, model_name, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='#111111')

    axes[0].plot(history_dict['accuracy'], color='#8B0000', linewidth=2.5, label='Train Accuracy')
    axes[0].plot(history_dict['val_accuracy'], color='white', linewidth=2.5, label='Val Accuracy')
    axes[0].set_title(f'{model_name} Accuracy', fontsize=14)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    axes[1].plot(history_dict['loss'], color='#8B0000', linewidth=2.5, label='Train Loss')
    axes[1].plot(history_dict['val_loss'], color='white', linewidth=2.5, label='Val Loss')
    axes[1].set_title(f'{model_name} Loss', fontsize=14)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.2)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(cnn_history, 'Custom CNN', os.path.join(PLOTS_DIR,'04_custom_cnn_training_history.png'))
plot_training_history(eff_history, 'EfficientNetB0', os.path.join(PLOTS_DIR,'05_efficientnetb0_training_history.png'))
plot_training_history(mob_history, 'MobileNetV2', os.path.join(PLOTS_DIR,'06_mobilenetv2_training_history.png'))
plot_training_history(inc_history, 'InceptionV3', os.path.join(PLOTS_DIR,'06b_inceptionv3_training_history.png'))


These plots reveal learning dynamics rather than just final numbers. A small train-validation gap usually suggests healthy generalization, while a widening gap indicates overfitting.

For evaluators, these curves show whether the model improved steadily, plateaued, or became unstable during fine-tuning.


In [ ]:
# Step 12 — Test set evaluation helpers
def get_true_labels(ds):
    y_true = np.concatenate([y.numpy() for x, y in ds], axis=0)
    return np.argmax(y_true, axis=1), y_true

def evaluate_model(model, ds, model_name, cm_save_path):
    test_loss, test_acc = model.evaluate(ds, verbose=0)
    preds = model.predict(ds, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true_idx, y_true_onehot = get_true_labels(ds)

    report = classification_report(y_true_idx, y_pred, target_names=CLASS_NAMES, output_dict=True)
    cm = confusion_matrix(y_true_idx, y_pred)

    plt.figure(figsize=(7, 6), facecolor='#111111')
    sns.heatmap(cm, annot=True, fmt='d', cmap='magma', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'Confusion Matrix — {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(cm_save_path, dpi=300, bbox_inches='tight')
    plt.show()

    return {
        'model_name': model_name,
        'test_loss': test_loss,
        'test_acc': test_acc,
        'y_true': y_true_idx,
        'y_true_onehot': y_true_onehot,
        'y_pred': y_pred,
        'y_probs': preds,
        'report': report,
        'cm': cm
    }

cnn_eval = evaluate_model(custom_cnn, test_ds_norm, 'Custom CNN', os.path.join(PLOTS_DIR,'07_confusion_matrix_custom_cnn.png'))
eff_eval = evaluate_model(efficientnet_model, test_ds, 'EfficientNetB0', os.path.join(PLOTS_DIR,'08_confusion_matrix_efficientnet.png'))
mob_eval = evaluate_model(mobilenet_model, test_ds, 'MobileNetV2', os.path.join(PLOTS_DIR,'09_confusion_matrix_mobilenet.png'))
inc_eval = evaluate_model(inception_model, test_ds, 'InceptionV3', os.path.join(PLOTS_DIR,'09b_confusion_matrix_inception.png'))


In [ ]:
# Display classification reports
for eval_result in [cnn_eval, eff_eval, mob_eval, inc_eval]:
    print('\n' + '='*80)
    print(f"Classification Report — {eval_result['model_name']}")
    print('='*80)
    print(classification_report(eval_result['y_true'], eval_result['y_pred'], target_names=CLASS_NAMES))


The classification report gives a clinically richer view than accuracy alone. In this project, **recall for glioma** is especially important because missing a malignant tumor is usually more harmful than creating a false positive alert.

Confusion matrices add another layer of interpretability by showing exactly which tumor types are being confused with each other.


In [ ]:
# ROC-AUC curves
def plot_multiclass_roc(eval_results, save_path=os.path.join(PLOTS_DIR,'10_roc_curves.png')):
    plt.figure(figsize=(12, 9), facecolor='#111111')

    for model_result in eval_results:
        y_true = model_result['y_true_onehot']
        y_probs = model_result['y_probs']
        model_name = model_result['model_name']

        for i, class_name in enumerate(CLASS_NAMES):
            fpr, tpr, _ = roc_curve(y_true[:, i], y_probs[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=2, alpha=0.55, label=f'{model_name} - {class_name} (AUC={roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('One-vs-Rest ROC Curves')
    plt.legend(fontsize=8, loc='lower right')
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

plot_multiclass_roc([cnn_eval, eff_eval, mob_eval, inc_eval])


ROC-AUC curves quantify how well each class can be separated from the rest across decision thresholds. This matters because a classifier used in healthcare may eventually be tuned for higher sensitivity or higher specificity depending on clinical workflow.

For glioma screening, a threshold favoring recall may be more appropriate than one optimized only for balanced overall accuracy.


In [ ]:
# Step 13 — Grad-CAM helpers
def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
        if hasattr(layer, 'layers'):
            for sublayer in reversed(layer.layers):
                if isinstance(sublayer, tf.keras.layers.Conv2D):
                    return sublayer.name
    raise ValueError("No Conv2D layer found.")


def make_gradcam_heatmap_tfkerasvis(model, image_array, class_index):
    gradcam = Gradcam(model, model_modifier=ReplaceToLinear(), clone=True)
    score = CategoricalScore([class_index])
    cam = gradcam(score, image_array, penultimate_layer=-1)
    heatmap = np.uint8(255 * cam[0])
    return heatmap


def overlay_heatmap_on_image(image, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap_color = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(cv2.cvtColor((image*255).astype(np.uint8), cv2.COLOR_RGB2BGR), 1-alpha, heatmap_color, alpha, 0)
    overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    return overlay


In [ ]:
# Collect 2 sample images per class from test set
gradcam_samples = {}
for class_name in CLASS_NAMES:
    class_dir = os.path.join(TEST_DIR, class_name)
    files = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    gradcam_samples[class_name] = random.sample(files, min(2, len(files)))


best_gradcam_model = efficientnet_model  # update after comparison if needed


for class_name in CLASS_NAMES:
    fig, axes = plt.subplots(2, 2, figsize=(10, 8), facecolor="#111111")
    axes = axes.flatten()
    for idx, img_path in enumerate(gradcam_samples[class_name]):
        raw_img = preprocess_image(img_path, IMG_SIZE)
        input_img = np.expand_dims((raw_img * 255.0).astype(np.float32), axis=0)
        pred_probs = best_gradcam_model.predict(input_img, verbose=0)[0]
        pred_class = int(np.argmax(pred_probs))
        heatmap = make_gradcam_heatmap_tfkerasvis(best_gradcam_model, input_img, pred_class)
        overlay = overlay_heatmap_on_image(raw_img, heatmap)


        axes[idx*2].imshow(raw_img)
        axes[idx*2].set_title(f"Original — {class_name}", color="white")
        axes[idx*2].axis("off")


        axes[idx*2+1].imshow(overlay)
        axes[idx*2+1].set_title(f"Grad-CAM — Pred: {CLASS_NAMES[pred_class]}", color="white")
        axes[idx*2+1].axis("off")


    plt.tight_layout()
    plt.savefig(f"os.path.join(PLOTS_DIR,12_gradcam_{class_name}.png") if class_name=="glioma" else
                f"os.path.join(PLOTS_DIR,13_gradcam_{class_name}.png") if class_name=="meningioma" else
                f"os.path.join(PLOTS_DIR,14_gradcam_{class_name}.png") if class_name=="pituitary" else
                f"os.path.join(PLOTS_DIR,15_gradcam_{class_name}.png"), dpi=300, bbox_inches="tight")
    plt.show()


Grad-CAM helps answer a critical trust question: **what part of the MRI is the model using to make its prediction?** In healthcare AI, explainability does not replace clinical validation, but it can reveal whether the model is focusing on plausible anatomical regions rather than irrelevant artifacts.

This makes Grad-CAM valuable for model debugging, clinician communication, and broader regulatory confidence in explainable AI systems.


## PHASE 7 — Model Comparison

After training and evaluation, the next goal is to compare models on both technical and deployment dimensions. In a real hospital workflow, the best model is not always the largest one; the right choice depends on the balance between accuracy, robustness, inference cost, and trustworthiness.


In [ ]:
# Step 14 — Model comparison table
def count_params_millions(model):
    return round(model.count_params() / 1_000_000, 3)

comparison_rows = []
for model_obj, eval_obj, train_time, best_for in [
    (custom_cnn, cnn_eval, cnn_train_time, "Baseline"),
    (efficientnet_model, eff_eval, eff_train_time, "Accuracy"),
    (mobilenet_model, mob_eval, mob_train_time, "Speed"),
    (inception_model, inc_eval, inc_train_time, "Robustness")
]:
    macro_precision = eval_obj["report"]["macro avg"]["precision"]
    macro_recall = eval_obj["report"]["macro avg"]["recall"]
    macro_f1 = eval_obj["report"]["macro avg"]["f1-score"]
    comparison_rows.append({
        "Model": eval_obj["model_name"],
        "Test Accuracy": round(eval_obj["test_acc"], 4),
        "Precision": round(macro_precision, 4),
        "Recall": round(macro_recall, 4),
        "F1-Score": round(macro_f1, 4),
        "Parameters (M)": count_params_millions(model_obj),
        "Training Time (min)": round(train_time / 60, 2),
        "Best For": best_for
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv("models/model_comparison.csv", index=False)

comparison_df.style.highlight_max(subset=["Test Accuracy", "Precision", "Recall", "F1-Score"], color="lightgreen")                   .highlight_min(subset=["Parameters (M)", "Training Time (min)"], color="#FFDDC1")


In [ ]:
# Visual comparison bar chart
plt.figure(figsize=(10, 6), facecolor="#111111")
sorted_df = comparison_df.sort_values("Test Accuracy", ascending=False)
sns.barplot(data=sorted_df, x="Model", y="Test Accuracy", palette="rocket")
plt.title("Model Comparison by Test Accuracy")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR,"11_model_comparison_bar.png"), dpi=300, bbox_inches="tight")
plt.show()

comparison_df


This comparison table makes the project evaluator’s job easy: it puts model quality, parameter count, and training time in one place. That is the kind of artifact recruiters appreciate because it shows engineering maturity rather than single-metric tunnel vision.

The best deployment choice should be based not only on global accuracy, but also on class-specific clinical behavior, especially for glioma recall and F1-score.


In [ ]:
# Step 15 — Select best model
model_eval_map = {
    "Custom CNN": (custom_cnn, cnn_eval),
    "EfficientNetB0": (efficientnet_model, eff_eval),
    "MobileNetV2": (mobilenet_model, mob_eval),
    "InceptionV3": (inception_model, inc_eval)
}

best_row = comparison_df.sort_values(["F1-Score", "Test Accuracy"], ascending=False).iloc[0]
best_model_name = best_row["Model"]
best_model_obj, best_model_eval = model_eval_map[best_model_name]

print("Selected Best Model:", best_model_name)

best_model_obj.save(os.path.join(MODELS_DIR,"best_model_savedmodel")
best_model_obj.save(os.path.join(MODELS_DIR,"best_model.h5")

meta = {
    "best_model": f"{best_model_name.lower().replace(' ', '').replace('v', 'v')}_best.h5" if best_model_name != "Custom CNN" else "custom_cnn_best.h5",
    "best_accuracy": round(float(best_row["Test Accuracy"]), 3),
    "classes": CLASS_NAMES,
    "input_size": [224, 224, 3],
    "trained_on": "Brain Tumor MRI Dataset (Kaggle)",
    "training_date": "2026-06-18"
}

with open(os.path.join(MODELS_DIR, "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

meta


The final model should be selected with **clinical reasoning**, not just leaderboard ranking. For example, if EfficientNetB0 achieves the strongest F1-score and especially high recall for **glioma**, it is a strong deployment candidate because that class carries the highest malignancy risk among the four target categories.

That is a more convincing justification than simply saying it had the highest accuracy.


## PHASE 8 — Streamlit Deployment Preparation

The notebook now transitions from training to deployment. The production objective is to package saved models, metadata, evaluation plots, and explainability outputs so they can be surfaced cleanly in a polished Streamlit application.


## PHASE 9 — Artifact Checklist

At this point, the following should exist:
- `models/*.h5`
- `models/model_comparison.csv`
- `models/meta.json`
- `plots/*.png`

These files are the bridge between notebook-based experimentation and deployment-grade presentation.


## Manual Saving and Download

Because this version is prepared for manual Colab upload without Google Drive, all generated models, plots, and metadata are saved under local folders in the runtime. Before closing the session, run the next cell to package everything into one ZIP file and download it to your machine.


In [ ]:
# Final artifact export and manual download (no Drive required)
save_training_artifacts_summary()
zip_path = export_all_artifacts()
print('Artifact zip ready at:', zip_path)
print('Run the next line to download everything to your local machine:')
print("files.download('exports/brain_tumor_mri_artifacts.zip')")
# Uncomment to download immediately:
# files.download(zip_path)


## Key Insights & Conclusions

1. **Transfer learning is expected to outperform the custom CNN** on this dataset because pretrained backbones bring rich generalizable visual features and need fewer task-specific samples to achieve strong discrimination.
2. **Class-aware evaluation matters more than raw accuracy** in medical imaging. Glioma recall and F1-score are especially important because a missed malignant tumor is clinically more dangerous than a false positive.
3. **Grad-CAM improves trustworthiness** by showing whether the model focuses on relevant MRI regions instead of shortcuts or irrelevant artifacts.
4. **EfficientNetB0 is likely to be the best deployment candidate** when it balances strong predictive performance with robust fine-tuning behavior.
5. **MobileNetV2 remains highly valuable** when faster inference or lighter deployment footprint is a priority.
6. For a hospital CTO or evaluator, the recommended deployment path is:
   - Use the best transfer learning model as the default classifier
   - Keep Grad-CAM visible in the app for transparency
   - Position the tool as a **decision-support system**, not an autonomous diagnostic replacement
   - Validate externally before any real clinical rollout
